# 실습: MNIST와 CNN 분류
[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimhaneul050126-cmyk/codeplant-ai-vision-lab/blob/main/01_MNIST_CNN_실습.ipynb)

손글씨 숫자 사진을 숫자 배열로 바꾸고, 작은 CNN을 직접 학습합니다.

| 순서 | 내용 |
|---|---|
| 실습 1 | 데이터 준비 (훈련 · 검증 · 테스트) |
| 실습 2 | 이미지와 픽셀 확인 |
| 실습 3 | CNN 설계도 만들기 |
| 실습 4 | 학습하고 손실 관찰하기 |
| 실습 5 | 분석: 학습 곡선 · 오답 · 혼동 행렬 · 특징 맵 |
| 도전 | 한 가지만 바꿔 보는 실험 |
| 실습 6 | 최종 테스트 |
| 실습 7 | 내 손글씨 예측 (선택) |

위에서부터 셀을 차례로 실행하세요. GPU는 필요 없습니다. (런타임 → 모두 실행도 가능)

## 실습 1 · 데이터 준비
훈련 12,000장, 검증 5,000장, 테스트 10,000장을 사용합니다. 밝기를 255로 나눠 0~1로 바꾸고, 흑백 채널 1개를 붙입니다.

In [ ]:
# Colab에서 한글 글자가 깨지지 않도록 글꼴을 설치합니다 (처음 한 번, 약 20초)
import os, matplotlib
if not os.path.exists("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
    os.system("apt-get -qq install -y fonts-nanum > /dev/null")
import matplotlib.font_manager as fm
for f in fm.findSystemFonts():
    if "Nanum" in f:
        fm.fontManager.addfont(f)
matplotlib.rcParams["font.family"] = "NanumGothic"
matplotlib.rcParams["axes.unicode_minus"] = False

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
keras.utils.set_random_seed(SEED)

(x_all, y_all), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

# 빠른 수업 진행을 위해 일부만 사용합니다
x_train_raw, y_train = x_all[:12000], y_all[:12000]
x_val_raw,   y_val   = x_all[12000:17000], y_all[12000:17000]

def prepare(x):
    x = x.astype("float32") / 255.0      # 0~255 → 0~1
    return x[..., np.newaxis]              # (N, 28, 28) → (N, 28, 28, 1)

x_train, x_val, x_test = prepare(x_train_raw), prepare(x_val_raw), prepare(x_test_raw)
print("훈련:", x_train.shape, " 검증:", x_val.shape, " 테스트:", x_test.shape)

## 실습 2 · 이미지와 픽셀 확인
`INDEX`를 바꿔 다른 숫자를 확인해 보세요. 오른쪽은 같은 사진을 숫자로 본 모습입니다 (0 = 검정, 255에 가까울수록 밝음).

In [ ]:
INDEX = 0

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(x_train_raw[INDEX], cmap="gray")
axes[0].set_title(f"정답 라벨: {y_train[INDEX]}")
axes[0].axis("off")

crop = x_train_raw[INDEX][4:18, 4:18]       # 가운데 14×14만 확대
axes[1].imshow(crop, cmap="gray")
for r in range(crop.shape[0]):
    for c in range(crop.shape[1]):
        v = crop[r, c]
        axes[1].text(c, r, v, ha="center", va="center", fontsize=6,
                     color="black" if v > 128 else "white")
axes[1].set_title("가운데 부분의 픽셀 값")
axes[1].axis("off")
plt.show()
print("한 장의 픽셀 수:", x_train_raw[INDEX].size)

## 실습 3 · CNN 설계도
`Conv2D(8, 3)` + ReLU → 특징 맵 8장 → `MaxPooling2D(2)` → 14×14×8 → `Flatten` → `Dense(10)` → 숫자 0~9의 점수 10개

설계도는 함수로 만들어 두었습니다. **도전 실험**에서 값 하나만 바꿔 같은 함수로 새 모델을 만듭니다.

In [ ]:
FILTERS = 8          # 합성곱 필터 수
ACTIVATION = "relu"  # 활성화 함수
OPTIMIZER = "adam"   # 최적화기
EPOCHS = 3           # 훈련 데이터를 몇 번 볼지
BATCH_SIZE = 64

def build_model(filters=FILTERS, activation=ACTIVATION):
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(filters, 3, padding="same", activation=activation, name="conv"),
        keras.layers.MaxPooling2D(2, name="pool"),
        keras.layers.Flatten(name="flatten"),
        keras.layers.Dense(10, name="scores"),   # softmax 전 점수(logits)
    ])
    return model

def compile_model(model, optimizer=OPTIMIZER):
    model.compile(optimizer=optimizer,
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
    return model

model = compile_model(build_model())
model.summary()

**확인** · 첫 합성곱층의 매개변수는 3×3×1×8 + 8 = 80개입니다. `summary()`의 숫자와 같은지 보세요.

## 실습 4 · 학습하고 손실 관찰하기
한 배치마다 예측 → 손실 계산 → 역전파로 기울기 계산 → 가중치 수정이 일어납니다.

- `loss`, `accuracy`: 훈련 데이터 결과
- `val_loss`, `val_accuracy`: 검증 데이터 결과 (검증 데이터로는 가중치를 고치지 않습니다)

In [ ]:
steps = int(np.ceil(len(x_train) / BATCH_SIZE))
print(f"훈련 {len(x_train)}장 ÷ 배치 {BATCH_SIZE} → 1 에포크 = {steps} 스텝")

history = model.fit(x_train, y_train,
                    validation_data=(x_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)

## 실습 5 · 분석
### 5-1 학습 곡선
훈련 손실은 줄어드는데 검증 손실이 다시 늘어나면 **과적합**을 의심합니다.

In [ ]:
def plot_history(h, title=""):
    ep = range(1, len(h.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(ep, h.history["loss"], "o-", label="훈련")
    axes[0].plot(ep, h.history["val_loss"], "o-", label="검증")
    axes[0].set_title("손실 (낮을수록 좋음)"); axes[0].set_xlabel("에포크"); axes[0].legend()
    axes[1].plot(ep, h.history["accuracy"], "o-", label="훈련")
    axes[1].plot(ep, h.history["val_accuracy"], "o-", label="검증")
    axes[1].set_title("정확도 (높을수록 좋음)"); axes[1].set_xlabel("에포크"); axes[1].legend()
    fig.suptitle(title); plt.show()

plot_history(history, "기본 모델")

### 5-2 검증 데이터의 오답 보기
두 번째로 높은 점수의 숫자도 함께 표시합니다. 어느 획에서 헷갈렸을지 생각해 보세요.

In [ ]:
val_scores = model.predict(x_val, verbose=0)
val_prob = tf.nn.softmax(val_scores).numpy()
val_pred = val_prob.argmax(axis=1)
wrong = np.where(val_pred != y_val)[0]
print(f"검증 {len(y_val)}장 중 오답 {len(wrong)}장")

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes.ravel(), wrong[:12]):
    top2 = val_prob[i].argsort()[-2:][::-1]
    ax.imshow(x_val_raw[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"정답 {y_val[i]} → 예측 {top2[0]}\n2순위 {top2[1]} ({val_prob[i][top2[1]]:.0%})", fontsize=9)
plt.tight_layout(); plt.show()

### 5-3 혼동 행렬
행은 정답, 열은 예측입니다. 대각선 밖의 큰 숫자가 자주 헷갈리는 짝입니다.

In [ ]:
cm = tf.math.confusion_matrix(y_val, val_pred, num_classes=10).numpy()
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
for r in range(10):
    for c in range(10):
        ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=8,
                color="white" if cm[r, c] > cm.max() / 2 else "black")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("예측"); ax.set_ylabel("정답"); ax.set_title("검증 데이터 혼동 행렬")
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
r, c = np.unravel_index(off.argmax(), off.shape)
print(f"가장 많이 헷갈린 짝: 정답 {r} → 예측 {c} ({off[r, c]}번)")

### 5-4 합성곱층 특징 맵
같은 손글씨에 필터 8개를 적용한 결과입니다. 밝은 곳은 그 필터의 반응이 큰 곳입니다.

In [ ]:
SAMPLE = 0
feature_model = keras.Model(model.inputs, model.get_layer("conv").output)
maps = feature_model.predict(x_val[SAMPLE:SAMPLE + 1], verbose=0)[0]
weights = model.get_layer("conv").get_weights()[0]   # (3, 3, 1, 필터 수)

n = maps.shape[-1]
fig, axes = plt.subplots(2, n + 1, figsize=(2 * (n + 1), 4.4))
axes[0, 0].imshow(x_val_raw[SAMPLE], cmap="gray"); axes[0, 0].set_title("입력")
axes[1, 0].axis("off")
for k in range(n):
    axes[0, k + 1].imshow(maps[..., k], cmap="magma"); axes[0, k + 1].set_title(f"특징 맵 {k + 1}")
    axes[1, k + 1].imshow(weights[..., 0, k], cmap="bwr"); axes[1, k + 1].set_title("필터 3×3", fontsize=9)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 도전 · 한 가지만 바꿔 보는 실험
**한 번에 한 가지만** 바꾸고 같은 검증 데이터로 비교합니다. 아래 `CHANGE`의 값 하나만 고쳐 실행하세요.

| 바꿀 것 | 예시 |
|---|---|
| 필터 수 | `{"filters": 4}` 또는 `{"filters": 16}` |
| 에포크 | `{"epochs": 1}` 또는 `{"epochs": 6}` |
| 최적화기 | `{"optimizer": "sgd"}` |
| 활성화 함수 | `{"activation": "sigmoid"}` 또는 `{"activation": "tanh"}` |

결과는 아래 표에 계속 쌓입니다. 우연에 의한 차이도 있다는 점을 기억하세요.

In [ ]:
CHANGE = {"filters": 4}

results = globals().get("results", [])
if not results:
    results.append({"실험": "기본", "검증 정확도": history.history["val_accuracy"][-1],
                    "매개변수 수": model.count_params()})

cfg = {"filters": FILTERS, "activation": ACTIVATION, "optimizer": OPTIMIZER, "epochs": EPOCHS}
cfg.update(CHANGE)
keras.utils.set_random_seed(SEED)
experiment = compile_model(build_model(cfg["filters"], cfg["activation"]), cfg["optimizer"])
exp_history = experiment.fit(x_train, y_train, validation_data=(x_val, y_val),
                             epochs=cfg["epochs"], batch_size=BATCH_SIZE, verbose=0)
results.append({"실험": str(CHANGE), "검증 정확도": exp_history.history["val_accuracy"][-1],
                "매개변수 수": experiment.count_params()})

print(f"{'실험':<28}{'검증 정확도':>10}{'매개변수 수':>12}")
for r in results:
    print(f"{r['실험']:<28}{r['검증 정확도']:>10.2%}{r['매개변수 수']:>12,}")
plot_history(exp_history, f"실험: {CHANGE}")

## 실습 6 · 최종 테스트
검증 결과를 보고 최종 모델을 **먼저** 고른 뒤, 테스트 데이터는 마지막에 한 번만 사용합니다.
테스트 결과를 보고 다시 설정을 바꾸면 독립적인 평가라는 의미가 약해집니다.

In [ ]:
final_model = model          # 실험 모델이 더 좋았다면: final_model = experiment

test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f"테스트 정확도: {test_acc:.2%}  (테스트 {len(y_test):,}장)")

## 실습 7 · 내 손글씨 예측 (선택)
흰 종이에 검은 펜으로 숫자 **하나**를 굵게 쓰고 사진을 올립니다.
사진을 흑백 반전하고 숫자 부분만 잘라 28×28로 줄이는 간단한 전처리라서, 조명·배경·글씨 크기에 따라 결과가 흔들릴 수 있습니다. 먼저 **전처리된 모습**이 MNIST와 비슷한지 확인하세요.

In [ ]:
from PIL import Image, ImageOps
import io

def preprocess_photo(raw_bytes):
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(raw_bytes))).convert("L")
    img = ImageOps.invert(img)                         # 흰 종이·검은 글씨 → 검은 배경·밝은 글씨
    a = np.array(img, dtype="float32")
    a = np.where(a > a.mean() + a.std(), a, 0)         # 배경 지우기
    ys, xs = np.nonzero(a)
    if len(xs) == 0:
        raise ValueError("숫자를 찾지 못했습니다. 더 굵게, 더 크게 써 주세요.")
    a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = a.shape; side = max(h, w)
    square = np.zeros((side, side), dtype="float32")
    square[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = a
    digit = Image.fromarray(square).resize((20, 20), Image.LANCZOS)   # MNIST처럼 20×20 + 여백 4
    canvas = np.zeros((28, 28), dtype="float32")
    canvas[4:24, 4:24] = np.array(digit)
    canvas = canvas / max(canvas.max(), 1e-6)
    return canvas[np.newaxis, ..., np.newaxis]

try:
    from google.colab import files
    uploaded = files.upload()
    raw = next(iter(uploaded.values()))
except ImportError:
    raw = None
    print("Colab에서 실행하면 사진 업로드 창이 열립니다.")

if raw:
    x_mine = preprocess_photo(raw)
    prob = tf.nn.softmax(final_model.predict(x_mine, verbose=0)[0]).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(x_mine[0, ..., 0], cmap="gray"); axes[0].set_title("전처리된 내 숫자"); axes[0].axis("off")
    axes[1].bar(range(10), prob); axes[1].set_xticks(range(10)); axes[1].set_title(f"AI의 답: {prob.argmax()}")
    plt.show()

## 정리
- 손글씨 분류는 **잘린 흑백 사진 한 장 → 숫자 하나**를 고르는 문제입니다.
- 신호등 인식은 **컬러 사진 속 위치와 종류**를 함께 찾아야 하므로 다음 실습(YOLO 객체 탐지)으로 이어집니다.
- MNIST에서 정확도가 높다고 실제 도로에서도 안전하다는 뜻은 아닙니다. 데이터·평가·검증이 함께 필요합니다.

참고 · 실험 구성(에포크·최적화기·활성화 함수 비교, 혼동 행렬)은 [MJY-ML/02_cnn_theory_mnist](https://github.com/Hwk040319/MJY-ML/blob/main/02_cnn_theory_mnist.ipynb)를 참고해 Keras로 옮겼습니다.